$$
\max \sum_{i=0}^{N} v_i x_i
$$

$$
\sum_{i=0}^{N} w_i x_i \le W
$$

$$
x_i \in \{0,1\} \quad \forall i
$$

**Sets:** $N$ items, $i \in \{0,\dots,N\}$  
**Parameters:** $v_i$ (value), $w_i$ (weight), $W$ (capacity)  
**Variables:** $x_i \in \{0,1\}$


In [1]:
from pyomo.environ import *
from ortools.sat.python import cp_model

In [10]:
items = [
    {"name": "Laptop", "w": 3, "v": 2000},
    {"name": "Voucher", "w": 0, "v": 100},
    {"name": "Apple A", "w": 2, "v": 50},
    {"name": "Apple B", "w": 2, "v": 50},
    {"name": "Apple C", "w": 2, "v": 50},
    {"name": "Book", "w": 1, "v": 30},
    {"name": "Camera", "w": 4, "v": 1500},
]
capacity = 5
num_items = len(items)

In [11]:
def lp():
    model = ConcreteModel()
    model.I = Set(initialize=range(num_items))
    model.x = Var(model.I, domain=Binary)
    def obj_rule(model):
        return sum(items[i]["v"] * model.x[i] for i in model.I)
    
        
    def cap_rule(model):
        return sum(items[i]["w"] * model.x[i] for i in model.I) <= capacity
    model.obj = Objective(rule=obj_rule, sense=maximize)
    model.cap_cons = Constraint(rule=cap_rule)
    solver = SolverFactory("glpk")
    results = solver.solve(model, tee = False)
    print(value(model.obj))

In [62]:
def lp_preprocess_symbreak():
    # ---------------------------------------------------------
    # Note on Performance & Benchmark Results:
    # For simple 1D Knapsack, manual symmetry breaking and python-side preprocessing
    # might actually INCREASE total execution time (due to Pyomo's model-building overhead).
    # Solvers like GLPK/Gurobi natively handle 1D knapsack presolve in milliseconds.
    # However, manual symmetry breaking becomes crucial in complex NP-hard problems
    # (e.g., Bin Packing, VRPTW, TSP) where solvers struggle to detect multi-dimensional symmetries.
    # ---------------------------------------------------------
    preselected_items = []
    preselected_value = 0 
    filtered_items = []
    
    # ---------------------------------------------------------
    # Preprocessing: Remove items with w = 0 and v > 0
    # Removing these deterministic picks reduces the number of decision
    # variables and cuts down solver search-tree size.
    # ---------------------------------------------------------
    for item in items:
        if item['w'] == 0 and item['v'] > 0:
            preselected_items.append(item)
            preselected_value += item['v']
        else:
            filtered_items.append(item)
            
    # for symmetry break: now dont have to check all possible pairs 
    filtered_items = sorted( filtered_items, key=lambda item: (item["w"], item["v"]) ) 

    
    model = ConcreteModel()
    model.I = Set(initialize=range(len(filtered_items)))
    model.x = Var(model.I, domain=Binary)
    def obj_rule(model):
        return sum(filtered_items[i]["v"] * model.x[i] for i in model.I)
        
    def cap_rule(model):
        return sum(filtered_items[i]["w"] * model.x[i] for i in model.I) <= capacity
        
    model.obj = Objective(rule=obj_rule, sense=maximize)
    model.cap_cons = Constraint(rule=cap_rule)
    
    
    # ---------------------------------------------------------
    # Symmetry Breaking Constraint:
    # Enforces ordering on identical adjacent items (x[i+1] <= x[i]) to
    # eliminate redundant search branches.
    # Advanced solvers (e.g., Gurobi, CPLEX) have built-in presolve
    # symmetry breaking that may sometimes conflict with manual constraints.
    # ---------------------------------------------------------
    def symmetry_rule(model, i):
        """list of items has to be sorted"""
        if i < len(filtered_items) - 1:
            curr = filtered_items[i]
            next_item = filtered_items[i+1]
    
            if filtered_items[i]["w"] == filtered_items[i+1]["w"] and \
            filtered_items[i]["v"] == filtered_items[i+1]["v"]:
                return model.x[i] >= model.x[i+1]
        return Constraint.Skip
    
    
    # instead of sort items, can check all possible pairs.
    # def symmetry_rule(m, i, j):
    #     if i < j:
    #         item_i = filtered_items[i]
    #         item_j = filtered_items[j]
    
    #         if item_i["w"] == item_j["w"] and item_i["v"] == item_j["v"]:
    #             return m.x[j] <= m.x[i]
    
    #     return Constraint.Skip
    
    model.sym_cons = Constraint(model.I, rule = symmetry_rule)
    
    solver = SolverFactory("glpk")
    results = solver.solve(model, tee = False)
    
    
    total_value = value(model.obj) +  preselected_value
    print(total_value)

In [63]:
lp_preprocess_symbreak()

2150.0


In [37]:
model = cp_model.CpModel()

x = [model.NewBoolVar(f"x[{i}]") for i in range(num_items)]

weights = [] 
for i in range(num_items):
    weights.append(items[i]['w'] *x[i])

model.Add(sum(weights)<=capacity)

obj = [x[i]*items[i]['v'] for i in range(num_items)]

model.Maximize(sum(obj))


In [76]:
solver = cp_model.CpSolver()
# solver.parameters.max_time_in_seconds=1
results = solver.Solve(model)

In [77]:
solver.ObjectiveValue()

2150.0